In [0]:
# ============================================================
#  REGISTRO UNITY CATALOG PARA POWER BI
# ============================================================

STORAGE_ACCOUNT   = "stdatacolake"
CONTAINER_SILVER  = "silver-zone"
CONTAINER_CURRENT = "curated-zone"

BASE_SILVER  = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_CURRENT = f"abfss://{CONTAINER_CURRENT}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

In [0]:
# ── CREAR BASE DE DATOS ───────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS silver_db COMMENT 'Tablas limpias capa Silver'")
spark.sql("CREATE DATABASE IF NOT EXISTS current_db COMMENT 'KPIs y métricas para Power BI'")

print("✅ Bases de datos creadas")
spark.sql("SHOW DATABASES").show()

In [0]:
# ── REGISTRAR TABLAS SILVER ───────────────────────────────────
tablas_silver = {
    "vehiculos":  f"{BASE_SILVER}/vehiculos",
    "entregas":   f"{BASE_SILVER}/entregas",
    "bodegas":    f"{BASE_SILVER}/bodegas",
    "inventario": f"{BASE_SILVER}/inventario",
    "ventas":     f"{BASE_SILVER}/ventas",
    "visitas":    f"{BASE_SILVER}/visitas",
}

for nombre, ruta in tablas_silver.items():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS silver_db.{nombre}
        USING DELTA
        LOCATION '{ruta}'
    """)
    print(f"  ✅ silver_db.{nombre}")

In [0]:
# ── REGISTRAR TABLAS CURRENT (KPIs) ──────────────────────────
tablas_current = {
    "kpi_entregas_vehiculo":  f"{BASE_CURRENT}/kpi_entregas_vehiculo",
    "kpi_inventario_bodega":  f"{BASE_CURRENT}/kpi_inventario_bodega",
    "kpi_ventas_vendedor":    f"{BASE_CURRENT}/kpi_ventas_vendedor",
    "kpi_visitas_vendedor":   f"{BASE_CURRENT}/kpi_visitas_vendedor",
}

for nombre, ruta in tablas_current.items():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS current_db.{nombre}
        USING DELTA
        LOCATION '{ruta}'
    """)
    print(f"  ✅ current_db.{nombre}")

In [0]:
# ── VERIFICAR TODAS LAS TABLAS ────────────────────────────────
print("=== SILVER ===")
spark.sql("SHOW TABLES IN silver_db").show()

print("=== CURRENT ===")
spark.sql("SHOW TABLES IN current_db").show()

In [0]:
# ── VALIDACIÓN FINAL DE CONTEOS ───────────────────────────────
tablas_todas = {
    "silver_db.vehiculos":             None,
    "silver_db.entregas":              None,
    "silver_db.bodegas":               None,
    "silver_db.inventario":            None,
    "silver_db.ventas":                None,
    "silver_db.visitas":               None,
    "current_db.kpi_entregas_vehiculo": None,
    "current_db.kpi_inventario_bodega": None,
    "current_db.kpi_ventas_vendedor":   None,
    "current_db.kpi_visitas_vendedor":  None,
}

print(f"{'Tabla':<40} {'Filas':>10}")
print("-" * 52)
for tabla in tablas_todas:
    count = spark.sql(f"SELECT COUNT(*) as n FROM {tabla}").collect()[0]["n"]
    print(f"{tabla:<40} {count:>10,}")